# Model A on Kaggle: reference-conditioned pix2pix GAN (from scratch)

Settings: **GPU T4**, **Internet on** (VGG-19 and LPIPS weights are downloaded). No secret is needed.

In [ ]:
# ---- What this session does ----
# Attach as inputs: the data notebook's output (hydrochii/anime-colorization-data) + the newest previous
# version of THIS notebook when continuing (its gan/ checkpoints, repo/outputs and repo/results are used below).
#
#   session              SMOKE TRAIN | SAMPLE NODROP EVAL
#   v1  smoke + train      T     T   |   F      F     F     check the grid + it/s, then train until the time limit
#   v2.. train             F     T   |   F      F     F     resumes from the attached checkpoint until max_steps
#   last test outputs      F     F   |   T      T     T     gan_best.pt (best val LPIPS) on all three modes
#
# Every step skips work whose files already exist, so an unfinished session is continued by attaching its
# version as an input and running again.
RUN_SMOKE  = True    # 2,000 steps on 1,000 training images: wiring, sample grid, it/s on this GPU
RUN_TRAIN  = True    # full training (configs/gan.yaml), resumes from the newest attached checkpoint
RUN_SAMPLE = False   # test outputs for self_ref / cross_ref / shuffled_ref with dropout on (5 seeds)
RUN_NODROP = False   # ablation: dropout off at test time (gan_nodrop, self_ref): every seed gives the same image
RUN_EVAL   = False   # score every complete run in outputs/ that has no results CSV yet

# ---- Settings ----
REPO_URL = "https://github.com/hidrochin/IT5413-_Anime-line-art-colorization_Project.git"
BRANCH   = "feat/nhpthinh"
SESSION_MINUTES = 690          # Kaggle stops at 720 min; keep a margin for the final save
SMOKE_STEPS = 2000

In [ ]:
import glob, os, time
T0 = time.time()

!git clone -q -b {BRANCH} {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!git log -1 --oneline
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Unpack the data to /kaggle/temp (not part of the output).
TAR = glob.glob("/kaggle/input/**/AnimeFaceColorization.tar", recursive=True)[0]
!mkdir -p /kaggle/temp && tar -xf {TAR} -C /kaggle/temp
os.environ["ANIME_DATA_ROOT"] = "/kaggle/temp/AnimeFaceColorization"
os.environ["ANIME_SPLITS_DIR"] = os.path.join(os.path.dirname(TAR), "splits")
print("data:", os.environ["ANIME_DATA_ROOT"], "| splits:", os.environ["ANIME_SPLITS_DIR"])
print(f"setup took {(time.time() - T0) / 60:.1f} min")

In [ ]:
# Model A's own files live in /kaggle/working (= the version's output); test outputs and metric CSVs use
# the repo defaults outputs/ and results/ (also under /kaggle/working/repo).
W = "/kaggle/working"
RUN_DIR = f"{W}/gan"             # checkpoints/, gan_best.pt, gan_G.pt, log.csv, samples/
SMOKE_DIR = f"{W}/gan_smoke"

def minutes_left(reserve=0):
    return int(SESSION_MINUTES - reserve - (time.time() - T0) / 60)

def newest(pattern):
    found = sorted(glob.glob(pattern, recursive=True))
    return found[-1] if found else None

def find_input_dir(rel, max_depth=5):
    """Shallowest /kaggle/input/<...>/<rel> directory (no '**', which would crawl thousands of PNGs)."""
    for depth in range(1, max_depth + 1):
        hits = sorted(d for d in glob.glob("/kaggle/input/" + "*/" * depth + rel) if os.path.isdir(d))
        if hits:
            return hits[-1]
    return None

# Attached previous version of this notebook (if any).
PREV_CKPT_DIR = find_input_dir("gan/checkpoints")
PREV_RUN_DIR = os.path.dirname(PREV_CKPT_DIR) if PREV_CKPT_DIR else None
PREV_OUTPUTS = find_input_dir("repo/outputs")
PREV_RESULTS = find_input_dir("repo/results")
print("attached checkpoints:", PREV_CKPT_DIR)
print("attached outputs/:   ", PREV_OUTPUTS)
print("attached results/:   ", PREV_RESULTS)

# Carry the previous run forward (log.csv, exported weights, samples): the training log keeps growing across
# sessions and sampling-only sessions find gan_best.pt here. Checkpoints come in through --init-from.
if PREV_RUN_DIR and not os.path.exists(f"{RUN_DIR}/log.csv"):
    !mkdir -p {RUN_DIR} && cp -rn {PREV_RUN_DIR}/log.csv {PREV_RUN_DIR}/gan_best.pt {PREV_RUN_DIR}/gan_G.pt {PREV_RUN_DIR}/samples {RUN_DIR}/ 2>/dev/null
if PREV_OUTPUTS:
    !mkdir -p outputs && cp -rn {PREV_OUTPUTS}/. outputs/
if PREV_RESULTS:
    !mkdir -p results && cp -rn {PREV_RESULTS}/. results/
!ls {RUN_DIR} outputs results 2>/dev/null

def export_eval_assets():
    if not os.path.exists("data/eval/gt"):
        !python -m scripts.export_eval_assets --out data/eval

def show_run(run_dir):
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import Image, display
    log_path = f"{run_dir}/log.csv"
    if not os.path.exists(log_path):
        print("no log at", log_path); return
    log = pd.read_csv(log_path)
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
    for k in ("loss_d", "loss_g_adv"):
        ax[0].plot(log["step"], log[k].rolling(10, min_periods=1).mean(), label=k)
    ax[0].set_xlabel("step"); ax[0].legend()
    ax[1].plot(log["step"], log["loss_l1"].rolling(10, min_periods=1).mean(), label="train L1")
    val = log.dropna(subset=["val_lpips"])
    ax[1].plot(val["step"], val["val_l1"], "o-", label="val L1")
    ax[1].plot(val["step"], val["val_lpips"], "o-", label="val LPIPS")
    ax[1].set_xlabel("step"); ax[1].legend()
    ax[2].plot(log["step"], log["it_s"]); ax[2].set_xlabel("step"); ax[2].set_ylabel("it/s")
    plt.show()
    grids = sorted(glob.glob(f"{run_dir}/samples/*.png"))
    if grids:
        print(grids[-1]); display(Image(grids[-1]))

## 1. Smoke run (1,000 training images)

Checks data → GPU warp → G/D → losses → val probe end to end and measures speed (`it_s` in the log). Grid
columns: sketch | reference | dropout seed 0 | seed 1 | ground truth. After ~2k steps the outputs should
follow the sketch lines with roughly the reference colors.

In [ ]:
if RUN_SMOKE:
    !python -m scripts.train_gan --config configs/gan.yaml --out {SMOKE_DIR} --limit 1000 --max-steps {SMOKE_STEPS}
    show_run(SMOKE_DIR)

## 2. Full training (resumable)

Resumes from the newest checkpoint in this session's `gan/checkpoints`, else from the attached previous version
(`--init-from`). Saves every 30 min and at the end; `gan_best.pt` is the generator with the best val LPIPS.
Set `train.max_steps` in `configs/gan.yaml` from the it/s printed by the smoke run (budget ≈ 15 GPU-h).

In [ ]:
if RUN_TRAIN:
    remaining = minutes_left(10)
    init = f"--init-from {PREV_CKPT_DIR}" if PREV_CKPT_DIR else ""
    print(f"training for up to {remaining} min {init}")
    !python -m scripts.train_gan --config configs/gan.yaml --out {RUN_DIR} --max-minutes {remaining} {init}
    show_run(RUN_DIR)

## 3. Test outputs (shared `scripts/sample.py`)

Uses `gan_best.pt`. Writes `outputs/gan/<mode>/<id>_s<seed>.png` (5 seeds, dropout on) and a row in
`results/speed.csv`. Rerunning skips images that already exist.

In [ ]:
CKPT = f"{RUN_DIR}/gan_best.pt"
print("weights:", CKPT, os.path.exists(CKPT))
SAMPLE = f"python -m scripts.sample --config configs/gan.yaml --set checkpoint={CKPT} --batch-size 32"
if RUN_SAMPLE:
    !{SAMPLE} --mode self_ref cross_ref shuffled_ref --max-minutes {minutes_left(30)}
if RUN_NODROP:
    !{SAMPLE} --mode self_ref --model gan_nodrop --set test_dropout=false --max-minutes {minutes_left(30)}
!cat results/speed.csv 2>/dev/null

## 4. Evaluate

Only runs whose PNG count is complete (every test id x 5 seeds) are scored.

In [ ]:
from src.data import MODE_SPLIT
from src.utils.seeding import SEEDS

def n_expected(mode):
    with open(os.path.join(os.environ["ANIME_SPLITS_DIR"], MODE_SPLIT[mode] + ".txt")) as f:
        return len([l for l in f if l.strip()]) * len(SEEDS)

if RUN_EVAL and os.path.isdir("outputs"):
    export_eval_assets()
    for run in sorted(os.listdir("outputs")):
        for mode in sorted(os.listdir(f"outputs/{run}")):
            n = len(glob.glob(f"outputs/{run}/{mode}/*.png"))
            if os.path.exists(f"results/{run}_{mode}.csv"):
                continue
            if n < n_expected(mode):
                print(f"{run}/{mode}: {n}/{n_expected(mode)} PNGs, not scored yet"); continue
            !python -m scripts.evaluate --model {run} --mode {mode}
    !for f in results/*_*_ref.csv; do echo "== $f"; cat "$f"; done

## 5. Before the session ends

Kept in the output: `gan/` (checkpoints, `gan_best.pt`, `gan_G.pt`, `log.csv`, `samples/`), `gan_smoke/`,
`repo/outputs/`, `repo/results/`. Copy `results/*.csv`, `gan/log.csv` and a few sample grids into the repo's
`results/` and commit them; never commit checkpoints or generated images.

In [ ]:
!du -sh {W}/gan {W}/gan_smoke {W}/repo/outputs {W}/repo/results 2>/dev/null
print(f"session time used: {(time.time() - T0) / 60:.0f} min")